# 05 · Alt/BTC ratio mean reversion

Walkthrough of a relative-value strategy: ten altcoins priced in Bitcoin, trading the ratio alt/BTC back to its average. This notebook explains, calls `src/` and plots. It decides nothing: all settings come from `config_ratio.yaml` (parameters were fixed on train data with `backtest.parameter_sweep`, shown again in section 7).

**Idea in one line:** Bitcoin is the anchor. An alt's fair value is `BTC price x mean(alt/BTC)`. When the alt is expensive in BTC we short it against BTC, when it is cheap we buy it against BTC, and we close when the ratio is back at its mean.

Run `python src/data.py config_ratio.yaml` once before this notebook to cache the daily Yahoo data in `data/`.

In [ ]:
import sys
import warnings
from pathlib import Path

# Make the repo root importable, whether Jupyter starts in notebooks/ or the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src import data, strategy, backtest, plots

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)
plots.apply_style()

cfg = data.load_config("config_ratio.yaml")
setup, params = cfg["setup"], cfg["parameters"]
BTC = data.coin_name(setup["numeraire"])

## 0. Load data and run the strategy once

Load the cleaned daily closes with `data.load_daily_prices`, estimate everything that needs history on train data only with `backtest.calibrate_pairs`, and run the configured portfolio on the train and test periods with `backtest.run_ratio_portfolio`. Every section below reads from these objects.

In [ ]:
prices = data.load_daily_prices(cfg)
train, test = data.split_train_test(prices, setup["train_end"])
calibration = backtest.calibrate_pairs(cfg, prices)

train_result, train_pairs = backtest.run_ratio_portfolio(cfg, prices, "train")
test_result, test_pairs = backtest.run_ratio_portfolio(cfg, prices, "test")

print(f"prices: {prices.index[0]:%Y-%m-%d} -> {prices.index[-1]:%Y-%m-%d}, {len(prices)} days, {prices.shape[1]} coins")
for period in ("train", "test"):
    start, end = backtest.period_bounds(cfg, period)
    print(f"{period:5s}: {start:%Y-%m-%d} -> {end:%Y-%m-%d} (end exclusive)")

## 1. Which products and why

**Universe:** Bitcoin (`BTC-USD`) as the numeraire plus ten alts. The candidates were the large, liquid coins listed on every major exchange. The binding constraint was **a long, complete, common Yahoo history**, because the train period needs to cover a full crypto cycle:

| Alt | Why it is in |
|---|---|
| ETH | Largest alt, deepest spot and perp markets; the cleanest "beta to BTC" case |
| BNB | Large exchange token, liquid on Binance spot and perps |
| XRP | Top-5 by market cap for most of the sample, high retail volume |
| ADA | Large-cap smart-contract platform, liquid since 2017 |
| DOGE | Very liquid but sentiment-driven: a deliberate stress case for mean reversion |
| LTC | One of the oldest alts, long BTC-like history |
| LINK | Largest oracle token, liquid perps |
| BCH | BTC fork: economically closest to BTC, natural ratio candidate |
| XLM | Payments coin, often moves with XRP; liquid since 2017 |
| TRX | Large-cap, long history, lower correlation with BTC |

**Dropped:** SOL, AVAX and DOT were also liquid but their Yahoo history only starts in 2020. Including them would cut the common sample in half and leave the train period without the 2018 bear market.

**Data:** daily adjusted closes from Yahoo Finance via `data.fetch_yahoo_daily` (00:00 UTC close, crypto trades every day). `data.coverage_report` compares every coin with the full calendar. Gaps of up to `max_ffill_days` would be forward-filled and longer gaps dropped. In this sample there are no gaps at all.

In [ ]:
data.coverage_report(cfg)

Normalised prices in USD (small multiples, log scale): all coins share the big crypto cycles, but their amplitudes differ a lot, which is what moves the ratios.

In [ ]:
plots.plot_prices(prices.loc[setup["start"]:]);

Correlation of daily log returns on **train data**, with `strategy.return_correlations`. Every alt co-moves with BTC (0.5 to 0.8), which is why BTC is a sensible hedge leg. The correlation is far from 1, so the ratio still carries a lot of risk of its own (section 3).

In [ ]:
plots.plot_correlation_matrix(strategy.return_correlations(train));

## 2. Trading frequency and time horizon

The horizon follows from how fast the ratio reverts. `strategy.estimate_half_life` fits an AR(1) to a series on train data and returns the half-life in days:

- `half_life_level`: half-life of the log ratio itself, measured against a single long-run mean. This is hundreds of days.
- `half_life_dev`: half-life of the log ratio's deviation from its 90-day rolling mean, which is what the signal trades. This is 23 to 55 days (3 to 8 weeks).

So the strategy is a **swing-trading horizon: weeks, not days or months**. The time stop in section 6 is `time_stop_halflives` x `half_life_dev`, capped at `max_holding_days`. With these half-lives, 3 half-lives is 70 to 165 days, so for every pair the **60-day cap is the binding time stop**.

In [ ]:
calibration[["half_life_level", "half_life_dev"]].round(1)

Realised frequency from the backtest trade logs (`backtest.trade_frequency`):

In [ ]:
pd.DataFrame({
    "train": backtest.trade_frequency(train_result.trades, cfg, "train"),
    "test": backtest.trade_frequency(test_result.trades, cfg, "test"),
}).round(1)

## 3. Data features and indicators

All features for one pair come from `strategy.ratio_features`. Each value on day *t* uses data up to day *t* only:

| Feature | Definition |
|---|---|
| ratio | alt / BTC: the alt's price in BTC |
| log ratio | `r_t = ln(alt_t) - ln(BTC_t)`. Its daily change is the return of a dollar-neutral long alt / short BTC position |
| mean, std | rolling mean and standard deviation of `r` over the last `zscore_lookback` days (`strategy.rolling_mean_std`) |
| z | `(r_t - mean_t) / std_t` (`strategy.zscore`) |
| fair value | `BTC_t x exp(mean_t)`: the alt price if the ratio were at its mean (`strategy.fair_value`) |
| deviation | `alt / fair value - 1` |

The z-score is used on the **log** ratio, because it is symmetric (+10% and -10% are equally far from the mean) and it maps directly onto the PnL of the BTC-hedged position.

In [ ]:
eth = strategy.ratio_features(prices["ETH"], prices[BTC], params["zscore_lookback"], params["zscore_method"])
eth.loc[setup["train_end"]:].head(8)

**What is the "average ratio"?** It is configurable (`zscore_method`):

- **rolling** (main): the mean over the last `zscore_lookback` days. It adapts to new regimes, but it also follows a trend, so a ratio that keeps drifting never looks very extreme.
- **expanding**: the mean of all history so far. It is stable, but after a regime change the ratio can sit far from it for years.

On ETH/BTC the difference is easy to see. The expanding mean barely moves while the ratio roughly halves and doubles.

In [ ]:
eth_expanding = strategy.ratio_features(prices["ETH"], prices[BTC], params["zscore_lookback"], "expanding")
plots.plot_mean_comparison(
    eth["ratio"].loc[setup["start"]:],
    {f"rolling mean ({params['zscore_lookback']}d)": eth["mean"].loc[setup["start"]:],
     "expanding mean": eth_expanding["mean"].loc[setup["start"]:]},
    title="ETH/BTC: two definitions of the average ratio",
);

Expanding vs rolling on the **train** period. The expanding variant hardly ever trades (12 trades vs 175). Its standard deviation contains every past trend, so it is wide and z only rarely gets beyond ±2 (for example ETH: 17 days on train vs 270 with the rolling mean). That gives fewer, longer bets on a long-run mean that, by the ADF test below, the ratios do not return to:

In [ ]:
expanding_train, _ = backtest.run_ratio_portfolio(cfg, prices, "train", {"zscore_method": "expanding"})
pd.DataFrame({"rolling (config)": train_result.metrics, "expanding": expanding_train.metrics}).loc[
    ["ann_return", "sharpe", "max_drawdown", "n_trades", "win_rate"]
]

**Is the ratio mean-reverting?** `strategy.pair_diagnostics` on train data. ADF p-values below 0.05 mean "mean-reverting":

- `adf_level`: the log ratio around one fixed mean. For 9 of 10 alts (all but TRX) it is **not** mean-reverting (p > 0.05). A fixed long-run average, as in a classic pairs setup, is not a credible target here.
- `adf_dev`: the deviation from the rolling mean is strongly mean-reverting. **Caveat:** removing a rolling mean makes almost any series look mean-reverting. This number justifies the *time scale* of the trades, not their profitability.
- `ratio_vol`: annualised volatility of the log ratio, 60% to 130%. The BTC hedge removes market direction, not most of the risk.

In [ ]:
calibration[["corr_btc", "ratio_vol", "adf_level", "adf_dev"]]

## 4. Entry and exit signals

`strategy.generate_signals` turns z into a raw direction, and `strategy.apply_risk` adds stops and sizing (section 6):

| Situation | Signal | Trade |
|---|---|---|
| z > +entry_z (alt expensive vs BTC) | -1 | **short alt, long BTC**, equal dollars |
| z < -entry_z (alt cheap vs BTC) | +1 | **long alt, short BTC**, equal dollars |
| short open and z <= exit_z, or long open and z >= -exit_z | 0 | close both legs (exit reason `mean`) |
| z jumps from one side to the other in one day | flip | close, then open the other side on the same day |

**Timing (no look-ahead):** z on day *t* uses closes up to *t*. The order is filled at the close of *t+1* (`execution_lag = 1`), so the first PnL is earned from *t+1* to *t+2*. Fees are charged at the fill.

In [ ]:
print({k: params[k] for k in ("zscore_lookback", "zscore_method", "entry_z", "exit_z")})

**Example 1: ETH/BTC on the test period.** The shaded band is the fair ratio +- `entry_z` standard deviations. Entries are marked with triangles (▲ long alt, ▼ short alt), exits with ✕, all on the signal day.

In [ ]:
d = test_pairs["ETH"].daily
plots.plot_ratio_bands(d["ratio"], d["mean"], d["std"], params["entry_z"], d["position"], title="ETH/BTC, test period");
plots.plot_spread_zscore(d["log_ratio"].rename("log ratio ETH/BTC"), d["z"], d["position"], params["entry_z"], params["exit_z"]);

**Example 2: BCH/BTC**, the best pair in the test period, and **example 3: BNB/BTC**, the worst. BNB kept getting cheaper against BTC (the ratio fell 38% over the test period). Every "BNB is cheap" long signal came too early, and all 5 long trades lost, closed by the stop-loss or the time stop. ETH/BTC above shows the same pattern: a ratio falling for two years produces a string of losing longs.

In [ ]:
for alt in ("BCH", "BNB"):
    d = test_pairs[alt].daily
    plots.plot_ratio_bands(d["ratio"], d["mean"], d["std"], params["entry_z"], d["position"], title=f"{alt}/BTC, test period");

## 5. Position sizing and management

- **Structure:** every trade is a **dollar-neutral pair**: the same USD notional long the cheap leg and short the expensive one. This expresses exactly the ratio view, and the position is flat to the overall crypto market direction. A pure alt-only position would mostly be a bet on crypto beta.
- **Equal risk per pair** (`strategy.equal_risk_sizes`): each pair's notional per leg is proportional to `1 / ratio_vol`, with the volatility estimated on train data only. Without this, DOGE (130% ratio vol) would carry about twice the risk of ETH (60%).
- **Leverage cap:** sizes are scaled so that with all 10 pairs open, the gross exposure (`sum |alt leg| + |BTC leg|`) is exactly `max_gross_leverage`. All 10 pairs may be open at the same time, so there is no further limit on concurrent positions.
- **No compounding, no intra-trade rebalancing of size:** notionals are fractions of the *initial* capital. The backtest holds a constant notional per day, which approximates daily rebalancing of both legs. The cost of that rebalancing is not charged (see limitations).
- **BTC legs are not netted** across pairs, a conservative choice: a long ETH/BTC and a short BNB/BTC would partly cancel in BTC in reality.

In [ ]:
calibration[["ratio_vol", "size"]].assign(notional_per_leg=calibration["size"] * setup["initial_capital"]).round(3)

In [ ]:
plots.plot_exposure(test_result.positions, params["max_gross_leverage"]);

## 6. Risk management

All rules live in `strategy.apply_risk` and are checked every day while a trade is open, in this order:

| Rule | Parameter | Why |
|---|---|---|
| reversion exit | `exit_z` | the profit target: the ratio is back at its mean |
| z-stop | `stop_z` | a sudden ratio breakout far beyond normal moves, so mean reversion has failed |
| stop-loss | `stop_loss` | the log ratio moved more than this against the trade since entry (about the loss as a fraction of notional). It catches **slow trends**, which the rolling mean follows so z never hits `stop_z` |
| take-profit | `take_profit` | off: the mean is the target |
| time stop | `time_stop_halflives` x pair half-life, capped by `max_holding_days` (the cap binds for every pair, see section 2) | a trade that has not reverted in a few half-lives probably will not |

After any stop the pair stays flat until the raw signal resets (back to 0 or to the other side). It does not re-enter the same move. An entry signal that fires while |z| is already beyond `stop_z` is treated the same way.

**Exposure limits:** the gross exposure can never exceed `max_gross_leverage` (section 5). There is no portfolio-level drawdown stop. It would be one more parameter fitted to a short history, so drawdowns are reported instead.

In [ ]:
print({k: params[k] for k in ("stop_z", "stop_loss", "take_profit", "time_stop_halflives", "max_holding_days")})
backtest.exit_reason_summary(test_result.trades)

How trades end on the test period:

- **Reversion exits work:** 51 trades, 98% winners, +67k in total.
- **Stop-losses cancel them out:** 61 trades (44% of all), -70k. Time stops (-10k) and z-stops (-5k) push the total below zero. The whole strategy comes down to "many reversions vs. a few more trends".
- **Stops slip:** the stop-loss is checked on daily closes and filled one day later. Stopped trades lost 12% of notional on average (worst -28%), against the 10% threshold.

## 7. Back-testing approach

| Choice | Setting |
|---|---|
| Periods | train `start -> train_end` for all decisions; test `train_end -> end` run **once** with the fixed config |
| Warm-up | rolling statistics use the days before each period (past data only). Every period starts flat |
| Calibration | half-lives and ratio vols (for sizing and the time stop) from data before `train_end` only (`backtest.calibrate_pairs`) |
| Execution | signal at close *t*, fill at close *t+1* |
| Accounting | constant notional per trade, PnL from daily simple returns of both legs, no compounding |
| Trading costs | `fee_bps_per_leg` + `slippage_bps_per_leg` on each leg at entry and exit |
| Shorting | `borrow_bps_per_day` on the short leg's notional every day: an explicit assumption for perp funding or margin borrow |
| Metrics | daily returns on initial capital, 365 days per year, risk-free rate 0 |

In [ ]:
pd.Series({k: v for k, v in setup.items() if k not in ("alts", "sweep_grid", "cost_multipliers")}, name="setup").to_frame()

**Parameter selection without overfitting.** The parameters were chosen once, on train only, from the grid in `setup.sweep_grid` (36 combinations). Below is the Sharpe ratio per combination, one panel per stop-loss level:

- **Every combination lost money on train.** Short windows (20 to 30 days) lose the most. Longer windows lose less, which says alt/BTC ratios tend to *trend* over days to weeks rather than revert.
- The configured cell (90 days, entry 2.0, stop-loss 0.10, Sharpe -0.08) is the least bad. The 90-day row is the best row in every panel, but the neighbours along `entry_z` are clearly worse (-0.54 and -0.67), so the exact cell is **fragile**. It was kept as an honest baseline, and the test period was not used to choose it.

In [ ]:
sweep = backtest.parameter_sweep(cfg, prices, setup["sweep_grid"], "train")
plots.plot_heatmap(sweep, x="entry_z", y="zscore_lookback", value="sharpe", facet="stop_loss");

**Sensitivity to costs** on the test period (`backtest.cost_sensitivity`): fee, slippage and borrow scaled together. Multiplier 0 is the gross result before any costs.

In [ ]:
backtest.cost_sensitivity(cfg, prices, setup["cost_multipliers"], "test")[
    ["ann_return", "sharpe", "max_drawdown", "profit_factor"]
]

## 8. PnL results

The strategy against two buy-and-hold benchmarks on the same capital (`backtest.performance_table`). The strategy is measured on initial capital without compounding. The benchmarks compound (CAGR, drawdown from peak).

In [ ]:
performance = backtest.performance_table({"train": train_result, "test": test_result}, cfg, prices)
performance[["total_return", "ann_return", "ann_vol", "sharpe", "sortino", "max_drawdown",
             "n_trades", "win_rate", "profit_factor", "avg_trade_pnl", "avg_holding_bars"]].astype(float)

Equity on the test period. The benchmarks dwarf the strategy. This is a market-neutral strategy with about 10% volatility, against buy-and-hold positions with 46 to 56% volatility in a strong period for BTC, so the comparison shows the opportunity cost rather than a like-for-like risk comparison.

In [ ]:
plots.plot_equity_curve({"strategy": test_result.equity, **backtest.benchmark_equity(cfg, prices, "test")});

The strategy on its own, train and test, with the drawdown on the test period:

In [ ]:
plots.plot_equity_curve({"strategy, train": train_result.equity, "strategy, test": test_result.equity});
plots.plot_drawdown(test_result.equity);

PnL per pair on the test period (`backtest.pnl_by_pair`):

In [ ]:
plots.plot_pnl_by_pair(backtest.pnl_by_pair(test_result.trades));

## 9. Trade walkthrough

The full trade log of the test period, one row per round trip. `signal_time` is the day the rule fired, and `entry_time` / `exit_time` are the fills one day later. `entry_ratio` / `exit_ratio` are alt/BTC at the fills, and `fair_ratio_at_entry` is the rolling mean the trade bets on. `pnl` is after fees, slippage and borrow.

In [ ]:
trade_log = test_result.trades[[
    "pair", "direction", "signal_time", "entry_time", "exit_time", "entry_z", "entry_ratio",
    "fair_ratio_at_entry", "exit_ratio", "notional", "pnl", "return", "costs", "holding_bars", "exit_reason",
]]
trade_log

Three trades in detail, chosen by rule with `backtest.pick_example_trades` (not by hand): a **typical reversion win** (the `mean` exit with median PnL), the **best** and the **worst** trade. Top panel: both legs rebased to 1 at the entry fill. Bottom panel: the z-score with the signal (○), the entry fill and the exit fill.

In [ ]:
examples = backtest.pick_example_trades(test_result.trades)
examples[["example", "pair", "direction", "entry_time", "exit_time", "entry_z", "exit_z", "pnl", "return", "exit_reason"]]

In [ ]:
for _, trade in examples.iterrows():
    alt = trade["pair"].split("/")[0]
    d = test_pairs[alt].daily
    plots.plot_trade(prices.loc[d.index, alt], prices.loc[d.index, BTC], d["z"], trade, padding=30);

**How to read them:**

- **Typical reversion win (short ADA / long BTC, Oct 2026):** ADA got expensive against BTC (z +2.1). The hedged short was filled a day later and closed after 11 days, when z crossed 0. The PnL is +12.9% of notional, after about 0.6% round-trip trading costs and the daily borrow.
- **Best trade (long BCH / short BTC, Feb-Mar 2024):** BCH was cheap at z -2.1 and then overshot to z +3.9 within 11 days, for +49.5% of notional. It is the biggest single contributor to BCH being the best pair.
- **Worst trade (long LTC / short BTC, Nov 2024):** LTC looked cheap at z -2.0, then BTC rallied about 30% in two weeks while LTC lagged. The 10% stop-loss fired on a daily close and was filled a day later, at -20.6%. Three weeks after the stop, LTC outran BTC: the stop was right about the risk and wrong about the outcome. Trades like this decide the overall result.

## 10. Limitations

- **Regime changes:** alt/BTC ratios trend for months in alt seasons and BTC-dominance phases. Over 2018-2026 the evidence (train sweep, test result) says the ratios trended more than they reverted at a 3-to-13-week horizon. Mean reversion is not a stable property of these ratios.
- **Survivorship bias:** the ten alts are coins that are still large in 2026. Coins that collapsed against BTC (and would have been "cheap" longs that never recovered) are missing, which flatters a long-the-laggard strategy.
- **Yahoo data quality:** a single composite close at 00:00 UTC, not a tradable exchange price. There are no bid/ask or volume checks, so slippage is an assumption. Very low-priced coins (DOGE, TRX) show a few days with no price change, which comes from rounding.
- **Shorting constraints:** shorting alts needs perps or margin. Funding rates vary a lot over time and by coin, so the flat `borrow_bps_per_day` is a simplification. Borrow can also become unavailable exactly when a short is crowded.
- **Stops on daily data:** stops are checked on daily closes and filled a day later, so realised stop losses are larger than the threshold (section 6). Intraday stops would be tighter but need intraday data.
- **Accounting simplifications:** a constant daily notional means daily rebalancing that is not charged. BTC legs are not netted, and there is no compounding.
- **Small sample of decisions:** 36 sweep combinations on one train period. Even the least-bad cell is fitted, and the test period has only about 140 trades.

## 11. Summary

| Question | Answer |
|---|---|
| Products | BTC as anchor + ETH, BNB, XRP, ADA, DOGE, LTC, LINK, BCH, XLM, TRX: large, liquid, complete Yahoo history since Nov 2017 |
| Horizon | weeks: deviation half-life 3 to 8 weeks, about 3.5 trades per pair per year (3 per month in total), average holding 19 to 24 days |
| Features | log ratio alt/BTC, 90-day rolling mean and std, z-score, fair value, half-life, ADF checks |
| Signals | short alt / long BTC at z > +2, long alt / short BTC at z < -2, exit at z = 0, filled the next day |
| Sizing | dollar-neutral pairs, equal risk (1 / ratio vol), gross exposure <= 2x |
| Risk | z-stop 4, stop-loss 0.10, time stop 60 days (the cap binds before 3 half-lives), no re-entry until a fresh signal |
| Backtest | train 2018-2022 for all decisions, test 2023-2026 once; fees, slippage and borrow included |
| Result | **no edge**: test -4.6% per year (Sharpe -0.44) after costs and +0.2% before costs; train -0.9% per year. BTC buy-and-hold made +49% per year on test |

**Next steps:** test the opposite (momentum) signal on train as a separate hypothesis, add a trend filter that skips pairs whose ratio is trending, use time-varying funding rates, and move to a perp-exchange data source with bid/ask.